### **Rôle principal du service :**

Ce fichier centralise la logique métier de création et de gestion des prescriptions médicales. Il orchestre l'assemblage des données du patient, des médicaments, du moteur de règles CDS (Clinical Decision Support) et du moteur d'IA (RAG) pour émettre une ordonnance sécurisée.

### **Étapes et processus implémentés :**

##### **Étape 1 : Validation initiale des données (create_prescription)**

- Le service commence par vérifier l'existence du patient dans la base de données (db.query(Patient)) en utilisant l'ID fourni. Si le patient n'existe pas, il lève une erreur.
- Il s'assure ensuite que tous les médicaments prescrits (drug_id) existent bien dans le catalogue (db.query(Drug.id)).

##### **Étape 2 : Création de l'ordonnance en base de données**

- Il crée l'objet principal Prescription rattaché au patient et au médecin, initialement avec le statut "draft" (brouillon).
- Il insère ensuite chaque médicament sous forme de PrescriptionLine (contenant la posologie, la voie d'administration, la durée, etc.).
- Il utilise db.flush() pour générer les identifiants (IDs) en base sans encore valider définitivement la transaction (commit).

##### **Étape 3 : Exécution du moteur de règles cliniques (CDS)**

- Il appelle la fonction analyse_prescription (définie dans cds_engine.py) en lui passant la prescription et le patient.
- Cette fonction retourne une liste brute d'alertes (alerts) basées sur des règles médicales strictes (allergies, âge, interactions ANSM, insuffisance rénale).

##### **Étape 4 : Enrichissement par l'Intelligence Artificielle (RAG)**

- Tolérance aux pannes (Best-effort) : Si le système IA (RAG) est activé, le service tente de l'appeler. Si le modèle d'IA plante ou est indisponible, le processus ne s'arrête pas (bloc try...except) et continue avec les alertes standard, garantissant que le médecin reçoit toujours ses alertes de base.
- Préparation du contexte : Avant d'appeler l'IA, le service prépare le terrain. Il extrait toutes les DCI des médicaments prescrits, charge les interactions potentielles depuis la base de données (DrugInteraction), et calcule l'âge du patient.
- Injection : Il appelle ensuite enrich_alerts_with_rag (défini dans ai_service.py) qui va injecter les explications médicales personnalisées (générées par Ollama ou Gemini) directement dans les objets d'alerte.

##### **Étape 5 : Sauvegarde finale et mise à jour du statut**

- Il insère toutes les alertes (qu'elles aient été enrichies par l'IA ou non) dans la base de données.
- Il met à jour le statut de la prescription : "alerts" si des dangers ont été détectés, ou "safe" si la prescription est propre.
- Il valide enfin la transaction complète en base de données via db.commit().

##### **Étape 6 : Fonctions de consultation (Lecture)**

- Le fichier inclut également des fonctions utilitaires pour lire les données, comme récupérer une ordonnance complète avec ses lignes et ses alertes (get_prescription), ou lister l'historique des ordonnances d'un patient donné (list_prescriptions_for_patient).